# Module 03 — State, Memory & Recovery (Colab)

NovaBridge's agent writes an **account summary** for each client. It serves many
tenants, runs overlap, and runs crash and get retried. Every run keeps *working
memory* while it thinks. The whole lab is one question:

> **When a run reaches into working memory, whose memory does it get?**

You will change exactly one function — `state_key(run_id, tenant)` — the key each
run uses to store its memory. You'll find an obvious fix that looks right, watch it
fail on recovery, and then find the fix that actually holds.

## 1. Set up (Postgres + recorded model outputs, ~2 min)

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/sanbhaumik/workshop-designing-data-infra-for-ai-agents.git repo
%cd repo

In [ ]:
!SKIP_OLLAMA=1 bash setup.sh
# Failsafe: ensure Python deps are installed even on PEP 668 runtimes.
!python -m pip install --break-system-packages -q -r requirements.txt

In [ ]:
import os
os.environ['NOVA_LLM'] = 'frozen'  # replay recorded real model outputs (deterministic)
os.environ['DATABASE_URL'] = 'postgresql://postgres@localhost:5432/nova'

## 2. Two tenants, one working memory

Two clients, different balances:

- **alpha** — Alpha Capital, USD 4,200,000
- **beta** — Beta Partners LP, USD 1,100,000

The agent keeps working memory in a shared `MemoryStore`, addressed by whatever key
`state_key(run_id, tenant)` returns. Here's the shipped version — it returns the
**same key for every run**:

In [ ]:
import sys, importlib, tempfile
from pathlib import Path
sys.path.insert(0, 'modules/03_state')

from nova.agent import Agent, MemoryStore
from nova.llm import get_llm
from nova.scheduler import Scheduler
from nova.store import get_store
from nova.trace import Tracer
import your_fix

llm = get_llm()
store = get_store(); store.init_schema()

def build(key):
    """Two agents (alpha + beta) sharing ONE MemoryStore, keyed by `key`."""
    memory = MemoryStore()
    tracer = Tracer(Path(tempfile.mkdtemp()) / 'trace.jsonl')
    return (Agent(store, llm, tracer, memory, key),
            Agent(store, llm, tracer, memory, key))

SCRIPT = ['A:read', 'A:reason', 'B:read', 'B:reason', 'A:save', 'B:save']
print(open('modules/03_state/your_fix.py').read())

### ✋ Predict #1

Alpha's run and Beta's run interleave through that one shared slot: Alpha reads and
reasons, then Beta reads and reasons (overwriting the slot), then Alpha saves.

**What will Alpha's saved summary contain — Alpha's balance, or Beta's?**

In [ ]:
store.reset_demo()
naive_key = lambda run_id, tenant: ''  # the shipped state_key: one slot for all
alpha, beta = build(naive_key)
Scheduler(SCRIPT).run(
    lambda: alpha.run_steps('alpha', 'run-a'),
    lambda: beta.run_steps('beta', 'run-b'),
)
print("Alpha's summary:", store.get_summary('alpha')['content'])

See the leaked row in the **real database** — Alpha's summary, holding Beta's balance:

In [ ]:
!psql "$DATABASE_URL" -c "SELECT client_id, content FROM summaries;"

### The leak

`state_key` returned `''` for every run, so `run-a` and `run-b` shared **one** memory
slot. Beta's run overwrote it, and Alpha saved from Beta's data. Two tenants, one
slot — a cross-tenant leak.

## 3. The obvious fix: give each run its own slot

Every run already has a `run_id`. Key memory on it and the runs stop colliding. Edit
the cell below (it writes `your_fix.py`):

In [ ]:
%%writefile modules/03_state/your_fix.py
def state_key(run_id: str, tenant: str) -> str:
    # Give each run its own memory slot.
    return run_id

In [ ]:
importlib.reload(your_fix)
store.reset_demo()
alpha, beta = build(your_fix.state_key)
Scheduler(SCRIPT).run(
    lambda: alpha.run_steps('alpha', 'run-a'),
    lambda: beta.run_steps('beta', 'run-b'),
)
print("Alpha's summary:", store.get_summary('alpha')['content'])
!python -m pytest modules/03_state/test_state.py::test_isolation_alpha_summary_has_no_beta_data -q

Clean, and the live-isolation test passes. Looks fixed. **Is it?**

### ✋ Predict #2 — recovery

Real runs crash and get retried, and schedulers **reuse attempt ids**. Here's the
sequence:

1. Alpha starts on attempt `slot-1`, reads, reasons — then the worker **crashes
   before saving**. Its half-done work sits in memory under `slot-1`.
2. The queue hands `slot-1` to **Beta**, which runs to completion.
3. Alpha's job is retried and **resumes from its checkpoint** at `slot-1`.

Your fix keys memory on `run_id`. **After the resume, will Alpha's summary be clean?**

In [ ]:
store.reset_demo()
alpha, beta = build(your_fix.state_key)
reused = 'slot-1'

# 1. Alpha crashes after reasoning, before saving.
alpha_run = alpha.run_steps('alpha', reused)
for step in alpha_run:
    if step == 'reason':
        break  # process dies here

# 2. Beta reuses the same attempt id and runs fully.
beta.run('beta', reused)

# 3. Alpha is retried and resumes from its checkpoint.
alpha.save_from_checkpoint('alpha', reused)

print('Alpha after resume:', store.get_summary('alpha')['content'])

In [ ]:
!python -m pytest modules/03_state/test_state.py -q

### The trap

The leak came back — through recovery. `run_id` felt like identity, but it's an
**ephemeral attempt id**: it gets reused, so a resumed run keyed on it loaded
*another tenant's* checkpoint. Isolation that holds while runs are live can still
break the moment a run is retried.

The durable key is the one thing that doesn't change across a crash and resume: the
**unit of work** — the tenant.

In [ ]:
%%writefile modules/03_state/your_fix.py
def state_key(run_id: str, tenant: str) -> str:
    # Key memory on the UNIT OF WORK, not the ephemeral attempt id.
    return tenant

In [ ]:
importlib.reload(your_fix)
!python -m pytest modules/03_state/test_state.py -q

### The aha

Isolation isn't "give each run its own slot." It's **key state on the identity of the
work, not the identity of the attempt.** Attempt ids are born, reused, and thrown away;
the unit of work persists across every retry and resume — so that's what memory must
be addressed by.

Run the full before/after, both scenarios at once:

In [ ]:
!python modules/03_state/compare.py

### Optional: run the real local model

Everything above replayed recorded outputs so the leak is reproducible. To watch the
same design play out against a live local model, install Ollama and set `NOVA_LLM=ollama`
(see `SETUP.md`). The design lesson is identical — only the wording of each summary changes.